In [2]:
#Verificar si tengo instaladas las librerías que voy a utilizar
import sqlalchemy

print(sqlalchemy.__version__)

2.0.43


In [3]:
import pyodbc

print(pyodbc.version)

5.3.0


In [8]:
#Llamar librería pandas y al archivo csv:
import pandas as pd

df = pd.read_csv("fda_clean.csv")

In [10]:
#Crear la conexión a SQL
from sqlalchemy import create_engine

In [12]:
#Cargar dataset y verificar
df = pd.read_csv(
    "fda_clean.csv"
)
df.head()

,report_id,receive_date,searched_drug,reaction,serious,death,hospitalization,patient_sex,patient_age,patient_weight,country,age_group
0,10003301,2014-02-28,IBUPROFEN,DYSPEPSIA,YES,NO,NO,FEMALE,NaN,NaN,NaN,UNKNOWN
1,10003301,2014-02-28,IBUPROFEN,RENAL IMPAIRMENT,YES,NO,NO,FEMALE,NaN,NaN,NaN,UNKNOWN
2,10003319,2014-03-12,IBUPROFEN,CHOLECYSTECTOMY,YES,NO,YES,FEMALE,46.0,NaN,US,ADULT
3,10003319,2014-03-12,IBUPROFEN,NEPHROLITHIASIS,YES,NO,YES,FEMALE,46.0,NaN,US,ADULT
4,10003319,2014-03-12,IBUPROFEN,BILIARY TRACT DISORDER,YES,NO,YES,FEMALE,46.0,NaN,US,ADULT


In [13]:
#Verificar nombres de columnas:
df.columns

Index(['report_id', 'receive_date', 'searched_drug', 'reaction', 'serious',
       'death', 'hospitalization', 'patient_sex', 'patient_age',
       'patient_weight', 'country', 'age_group'],
      dtype='object')

In [29]:
#Crear el motor de conexión:
server = "localhost\SQLEXPRESS"

database = "PharmaVigilanceDW"

connection_string = (

    "mssql+pyodbc://@"

    + server

    + "/"

    + database

    + "?driver=ODBC+Driver+17+for+SQL+Server"

)

engine = create_engine(connection_string)

In [30]:
#Probar conexión 
engine

Engine(mssql+pyodbc://@localhost\SQLEXPRESS/PharmaVigilanceDW?driver=ODBC+Driver+17+for+SQL+Server)

In [31]:
#Creo dimension drug
dim_drug = (

    df[["searched_drug"]]

    .drop_duplicates()

    .reset_index(drop=True)

)
#Renombrar la columna para que coincida en SQL
dim_drug = dim_drug.rename(
    columns={
        "searched_drug": "drug_name"
    }
)
#Creo IDs automaticos 
dim_drug["drug_id"] = (
    dim_drug.index + 1
)
#Reordeno columnas
dim_drug = dim_drug[
    ["drug_id", "drug_name"]
]
#Visualizo resultado
dim_drug.head()

,drug_id,drug_name
0,1,IBUPROFEN
1,2,NAPROXEN
2,3,ACETAMINOPHEN
3,4,TRAMADOL


In [32]:
#Cargo dimensión a SQL
dim_drug.to_sql(

    "dim_drug",

    con=engine,

    if_exists="append",

    index=False

)

C:\Users\luisi\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


4

In [37]:
#Creo dimension patient:
dim_patient = (

    df[
        [
            "patient_sex",
            "age_group",
            "country"
        ]
    ]

    .drop_duplicates()

    .reset_index(drop=True)

)
#Creo ID automático:
dim_patient["patient_id"] = (
    dim_patient.index + 1
)

#Reordeno columnas:
dim_patient = dim_patient[
    [
        "patient_id",
        "patient_sex",
        "age_group",
        "country"
    ]
]

#Visualizo resultado:
dim_patient.head()

,patient_id,patient_sex,age_group,country
0,1,FEMALE,UNKNOWN,NaN
1,2,FEMALE,ADULT,US
2,3,FEMALE,ADULT,DE
3,4,MALE,ADULT,US
4,5,MALE,PEDIATRIC,GB


In [39]:
#Cargo datos a SQL
dim_patient.to_sql(

    "dim_patient",

    con=engine,

    if_exists="append",

    index=False

)

186

In [52]:
#Creo dimensión reaction:
dim_reaction = (

    df[["reaction"]]

    .drop_duplicates()

    .reset_index(drop=True)

)

#Creo ID automático:
dim_reaction["reaction_id"] = (
    dim_reaction.index + 1
)

#Renombro columna:
dim_reaction = dim_reaction.rename(
    columns={
        "reaction": "reaction_name"
    }
)

#Reordeno columnas:
dim_reaction = dim_reaction[
    [
        "reaction_id",
        "reaction_name"
    ]
]

#Visualizo resultados:
dim_reaction.head()

,reaction_id,reaction_name
0,1,DYSPEPSIA
1,2,RENAL IMPAIRMENT
2,3,CHOLECYSTECTOMY
3,4,NEPHROLITHIASIS
4,5,BILIARY TRACT DISORDER


In [42]:
#Cargo datos a SQL
dim_reaction.to_sql(

    "dim_reaction",

    con=engine,

    if_exists="append",

    index=False

)

337

In [51]:
#Creo dimension severity:
dim_severity = (

    df[
        [
            "serious",
            "death",
            "hospitalization"
        ]
    ]

    .drop_duplicates()

    .reset_index(drop=True)

)

#Creo ID automática:
dim_severity["severity_id"] = (
    dim_severity.index + 1
)

#Reodeno columnas:
dim_severity = dim_severity[
    [
        "severity_id",
        "serious",
        "death",
        "hospitalization"
    ]
]

#Visualizo resultados:
dim_severity.head()


,severity_id,serious,death,hospitalization
0,1,YES,NO,NO
1,2,YES,NO,YES
2,3,NO,NO,NO
3,4,YES,YES,NO
4,5,YES,YES,YES


In [44]:
#Cargo datos a SQL
dim_severity.to_sql(

    "dim_severity",

    con=engine,

    if_exists="append",

    index=False

)

5

In [63]:
#Carga de datos a fact table: JOINs
#JOIN con dim_drug:
fact = df.merge(

    dim_drug,

    left_on="searched_drug",

    right_on="drug_name",

    how="left"

)

#JOIN con dim_patient
fact = fact.merge(

    dim_patient,

    on=[
        "patient_sex",
        "age_group",
        "country"
    ],

    how="left"

)

#JOIN con dim_reaction 
fact = fact.merge(

    dim_reaction,

    left_on="reaction",

    right_on="reaction_name",

    how="left"

)

#JOIN con dim_severity
fact = fact.merge(

    dim_severity,

    on=[
        "serious",
        "death",
        "hospitalization"
    ],

    how="left"

)


In [48]:
for col in fact.columns:
    print(col)

report_id
receive_date
searched_drug
reaction
serious
death
hospitalization
patient_sex
patient_age
patient_weight
country
age_group
drug_id
drug_name


In [49]:
fact.columns.tolist()

['report_id',
 'receive_date',
 'searched_drug',
 'reaction',
 'serious',
 'death',
 'hospitalization',
 'patient_sex',
 'patient_age',
 'patient_weight',
 'country',
 'age_group',
 'drug_id',
 'drug_name']

In [50]:
[col for col in fact.columns if "id" in col.lower()]

['report_id', 'drug_id']

In [53]:
dim_patient.head()

,patient_id,patient_sex,age_group,country
0,1,FEMALE,UNKNOWN,NaN
1,2,FEMALE,ADULT,US
2,3,FEMALE,ADULT,DE
3,4,MALE,ADULT,US
4,5,MALE,PEDIATRIC,GB


In [54]:
dim_reaction.head()

,reaction_id,reaction_name
0,1,DYSPEPSIA
1,2,RENAL IMPAIRMENT
2,3,CHOLECYSTECTOMY
3,4,NEPHROLITHIASIS
4,5,BILIARY TRACT DISORDER


In [55]:
dim_severity.head()

,severity_id,serious,death,hospitalization
0,1,YES,NO,NO
1,2,YES,NO,YES
2,3,NO,NO,NO
3,4,YES,YES,NO
4,5,YES,YES,YES


In [65]:
#Crear fact table 
fact_adverse_events = fact[
    [
        "report_id",
        "receive_date",
        "patient_age",
        "patient_weight",
        "drug_id",
        "patient_id",
        "reaction_id",
        "severity_id"
    ]
]

#Verifico
fact_adverse_events.head()

,report_id,receive_date,patient_age,patient_weight,drug_id,patient_id,reaction_id,severity_id
0,10003301,2014-02-28,NaN,NaN,1,1,1,1
1,10003301,2014-02-28,NaN,NaN,1,1,2,1
2,10003319,2014-03-12,46.0,NaN,1,2,3,2
3,10003319,2014-03-12,46.0,NaN,1,2,4,2
4,10003319,2014-03-12,46.0,NaN,1,2,5,2


In [70]:
#Guardo una copia csv para respaldo físico de ETL:
fact_adverse_events.to_csv(
    "fact_adverse_events.csv",
    index=False
)

In [71]:
#Cargo datos de fact table a SQL:
fact_adverse_events.to_sql(

    "fact_adverse_events",

    con=engine,

    if_exists="append",

    index=False

)

115

In [72]:
fact_adverse_events.isnull().sum()

report_id            0
receive_date         0
patient_age       2824
patient_weight    8528
drug_id              0
patient_id           0
reaction_id          0
severity_id          0
dtype: int64